# Nemotron: check the CPU export path before loading weights

**You are here → 06 — Nemotron with ExecuTorch.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| 0. Understand the request | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| 5. CPU portability (optional) | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| **→ 6. Compatibility study (optional) (current)** | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

Can the same **Nemotron 3 Nano 4B** hybrid checkpoint used in the serving chapters use the Qwen CPU deployment path? First test whether the installed ExecuTorch export recipe accepts its architecture.

Nemotron-H includes Mamba recurrent state alongside attention. Qwen's adapter does not supply that state mapping. The following cells execute a **fresh configuration-only probe** of the installed exporter. Unsupported configuration is a valid finding; this chapter does not run native inference or report generation timings.

Install the [notebook environment](../SETUP.md). Follow [the edge setup guide](../tutorial/EXECUTORCH_MAC.md), configure your edge Python, then run the cells in order. Downloads are opt-in; this check needs only the pinned `config.json`, not model weights.


## 1. Imports and editable configuration

`MODE="live"` performs the stages below. Choose **plan** to inspect without edge execution or **replay** to read the explicitly selected prior record. The notebook mode overrides an old local JSON mode. No stage installs packages or downloads weights implicitly.


In [1]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.edge_workflow import EdgeStudy

MODE = "live"  # "plan" previews stages; "replay" displays archived results
MODE = os.environ.get('LOCAL_AI_LAB_MODE', MODE)  # Explicit CPU validation override.
# Machine settings: tutorial/config/edge-nemotron.local.json
study = EdgeStudy('nemotron', root=ROOT, mode=MODE)
print('Effective mode:', study.mode)
print('Model:', 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16')
print('Hardware target: local CPU / separate ExecuTorch environment')
print('Session:', study.session_id)
print('Current-session measurements:', len(study.results))
display(JSON(study.config))


Effective mode: live
Model: nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16
Hardware target: local CPU / separate ExecuTorch environment
Session: 20261110T122034-8766f89a12
Current-session measurements: 0


<IPython.core.display.JSON object>

## 2. Preflight

Check the configured edge Python and installed runtime before any model work. Qwen requires a pinned source checkpoint and either a validated reusable export or explicit conversion/export. Nemotron needs only its pinned configuration. Missing prerequisites stop with instructions.


In [2]:
display(JSON(study.preflight()))


<IPython.core.display.JSON object>

## 3. Execute the compatibility probe

The next cell invokes [probe_nemotron_edge.py](../scripts/tutorial/probe_nemotron_edge.py) using your installed edge environment. It tests `nemotron_h` and `nemotron_3_nano_4b` against the installed model configuration enum and saves a unique result.

It reports exporter version, accepted model types and each attempt. If a recipe is accepted, conversion and native inference still require validation; acceptance alone supplies no performance result.


In [3]:
probe = study.probe()
display(JSON(probe))


<IPython.core.display.JSON object>

## 4. Interpret the finding

If the result is **blocked_compatibility**, the installed standard recipe lacks this model configuration. It does not establish that a custom port is impossible. A working port would need validated Mamba state handling, weight conversion, graph capture, supported operators, PTE loading and native generation before SQL or timing could be evaluated.


In [4]:
display(Markdown(study.report()))
display(JSON({'native_inference_run': probe.get('native_inference_run', False),
              'SQL_correctness': probe.get('sql_correctness'),
              'TTFT': probe.get('ttft_seconds'),
              'request_latency': probe.get('end_to_end_seconds'),
              'peak_memory': probe.get('peak_memory_bytes')}))


**Mode: live · Session: 20261110T122034-8766f89a12**

| Stage | Status | Evidence |
|---|---|---|
| probe | blocked_compatibility | experiment-tracker/tutorial/edge/sessions/20261110T122034-8766f89a12/probe-e8d2934e.json |

<IPython.core.display.JSON object>

## Cleanup

The edge stages execute in foreground child processes and retain no serving process. Save the notebook to retain genuine current outputs. Failures stay in the session evidence.


In [5]:
print(study.close())


Edge stages run in foreground child processes; no serving process is retained.


## Reference: previously measured results

The [edge guide](../tutorial/EXECUTORCH_MAC.md) preserves the previous Qwen native run and Nemotron compatibility check. Choose `MODE="replay"` explicitly to inspect those records. The saved artifacts and historical records remain unchanged; they are not results from a new Live execution.
